In [ ]:
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input 
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from sklearn.metrics import classification_report,confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import numpy as np

I0000 00:00:1789544693.895430   45007 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789544693.975559   45007 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789544696.445031   45007 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
training_path = "../Datasets/data/train/Medical/CT"
testing_path = "../Datasets/data/test/Medical/CT"
valid_path = "../Datasets/data/vaild/Medical/CT"

In [3]:
train_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range = 10,
    zoom_range = 0.2,
    horizontal_flip = True
)
test_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input
)
valid_dataug = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

In [4]:
training_set = train_dataug.flow_from_directory(
    training_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "binary"
)
valid_set = valid_dataug.flow_from_directory(
    valid_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "binary"
)
testing_set = test_dataug.flow_from_directory(
    testing_path,
    target_size = (224,224),
    batch_size = 16,
    class_mode = "binary",
    shuffle = False
)

Found 21065 images belonging to 2 classes.
Found 2657 images belonging to 2 classes.
Found 3989 images belonging to 2 classes.


In [5]:
training_set.class_indices

{'Chest CT': 0, 'kidney CT': 1}

In [17]:
base_model = EfficientNetV2B0(weights="imagenet",include_top=False,input_shape=(224,224,3))

In [18]:
base_model.trainable = False

In [19]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128,activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
output = Dense(1,activation="sigmoid")(x)

In [20]:
model = Model(inputs = base_model.input,outputs = output)

In [21]:
model.compile(optimizer=Adam(learning_rate=0.002),metrics=["accuracy"],loss="binary_crossentropy")

In [22]:
early_stop = EarlyStopping(monitor="loss",patience=3,restore_best_weights=True)

In [23]:
model.fit(training_set,validation_data=(valid_set),epochs=11,callbacks=[early_stop])

Epoch 1/11


I0000 00:00:1789103777.660308    1520 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1789103786.181676    2443 service.cc:153] XLA service 0x7e8d780375b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789103786.181730    2443 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1789103786.632408    2443 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789103789.138986    2443 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1789103789.725817    2443 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_22466__.265
I0000 00:00:1789103815.167885    2443 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 940/1317 ━━━━━━━━━━━━━━━━━━━━ 1:18 209ms/step - accuracy: 0.9832 - loss: 0.0405

I0000 00:00:1789104014.970307    2441 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_22466__.265


1317/1317 ━━━━━━━━━━━━━━━━━━━━ 375s 256ms/step - accuracy: 0.9963 - loss: 0.0117 - val_accuracy: 0.9974 - val_loss: 0.0077
Epoch 2/11
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 261s 198ms/step - accuracy: 0.9989 - loss: 0.0046 - val_accuracy: 0.9992 - val_loss: 0.0039
Epoch 3/11
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 226s 172ms/step - accuracy: 0.9991 - loss: 0.0053 - val_accuracy: 0.9989 - val_loss: 0.0024
Epoch 4/11
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 245s 186ms/step - accuracy: 0.9994 - loss: 0.0027 - val_accuracy: 0.9981 - val_loss: 0.0038
Epoch 5/11
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 197s 150ms/step - accuracy: 0.9992 - loss: 0.0063 - val_accuracy: 0.9977 - val_loss: 0.0101
Epoch 6/11
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 197s 150ms/step - accuracy: 0.9996 - loss: 0.0011 - val_accuracy: 1.0000 - val_loss: 2.4828e-04
Epoch 7/11
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 202s 153ms/step - accuracy: 0.9996 - loss: 0.0024 - val_accuracy: 0.9996 - val_loss: 0.0010
Epoch 8/11
1317/1317 ━━━━━━━━━━━━━━━━━━━━ 196s 149ms/step - accuracy:

In [24]:
prediction = model.predict(testing_set)

250/250 ━━━━━━━━━━━━━━━━━━━━ 49s 151ms/step


In [28]:
prediction

array([[3.240004e-39],
       [0.000000e+00],
       [0.000000e+00],
       ...,
       [1.000000e+00],
       [1.000000e+00],
       [1.000000e+00]], shape=(3989, 1), dtype=float32)

In [32]:
true_prediction = (prediction > .5).astype(int)

In [26]:
y_true = testing_set.classes

In [30]:
y_true

array([0, 0, 0, ..., 1, 1, 1], shape=(3989,), dtype=int32)

In [33]:
print(classification_report(y_true,true_prediction))
confusion_matrix(y_true,true_prediction)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2400
           1       1.00      1.00      1.00      1589

    accuracy                           1.00      3989
   macro avg       1.00      1.00      1.00      3989
weighted avg       1.00      1.00      1.00      3989



array([[2400,    0],
       [   0, 1589]])

In [34]:
model.save("CT_categors_model.keras")